In [3]:
import wave 
import pyroomacoustics as pa
import numpy as np
import scipy.signal as sp
import scipy

from calc_steering_vector import calculate_steering_vector

In [4]:
# 遅延和アレイ
def execute_dsbf(x, a):
    """
    x: (num_microphones, freq_bins, time_frames)
    a: (freq_bins, num_microphones)
    """
    s_hat = np.einsum("km,mkt->kt", np.conjugate(a), x)
    """s_hat: (freq_bins, time_frames)"""
    # ステアリングベクトルを掛ける
    c_hat = np.einsum("kt,km->mkt", s_hat, a)
    """c_hat: (num_microphones, freq_bins, time_frames)"""
    return c_hat

In [5]:
# MVDR
def execute_mvdr(x, y, a):
    """
    x: (num_microphones, freq_bins, time_frames)
    y: (num_microphones, freq_bins, time_frames)
    a: (freq_bins, num_microphones)
    """
    # 共分散行列を計算する
    Rcov = np.einsum("mkt,nkt->kmn", y, np.conjugate(y))
    """Rcov: (freq_bins, num_microphones, num_microphones)"""
    # 共分散行列の逆行列を計算する
    Rcov_inverse = np.linalg.pinv(Rcov)
    """Rcov_inverse: (freq_bins, num_microphones, num_microphones)"""
    # 分離フィルタを計算する
    Rcov_inverse_a = np.einsum("kmn,kn->km", Rcov_inverse, a) # 分子
    """Rcov_inverse_a: (freq_bins, num_microphones)"""
    a_H_Rcov_inverse_a = np.einsum("kn,kn->k", np.conjugate(a), Rcov_inverse_a) # 分母
    """a_H_Rcov_inverse_a: (freq_bins,)"""
    w_mvdr = Rcov_inverse_a / np.maximum(a_H_Rcov_inverse_a, 1.e-18)[:, None]
    """w_mvdr: (freq_bins, num_microphones)"""
    # 分離フィルタを掛ける
    s_hat = np.einsum("km,mkt->kt", np.conjugate(w_mvdr), x)
    """s_hat: (freq_bins, time_frames)"""
    # ステアリングベクトルを掛ける（マイクロホン入力信号中の目的音成分を推定）
    c_hat = np.einsum("kt,km->mkt", s_hat, a)
    """c_hat: (num_microphones, freq_bins, time_frames)"""
    return c_hat

In [16]:
# MaxSNR
def execute_max_snr(x, y):
    """
    x: (num_microphones, freq_bins, time_frames)
    y: (num_microphones, freq_bins, time_frames)
    """
    # 雑音の共分散行列
    Rn = np.average(np.einsum("mkt,nkt->ktmn", y, np.conjugate(y)), axis=1)
    """Rn: (freq_bins, num_microphones, num_microphones)"""
    # 入力共分散行列
    Rs = np.average(np.einsum("mkt,nkt->ktmn", x, np.conjugate(x)), axis=1)
    """Rs: (freq_bins, num_microphones, num_microphones)"""
    # 周波数の数を取得
    Nk = np.shape(Rs)[0]
    # 一般化固有値分解
    max_snr_filter = None
    for k in range(int(Nk)):
        w, v = scipy.linalg.eigh(Rs[k, ...], Rn[k, ...])
        """w: (num_microphones, ), v: (num_microphones, num_microphones)"""
        if max_snr_filter is None:
            max_snr_filter = v[None, :, -1]
        else:
            max_snr_filter = np.concatenate((max_snr_filter, v[None, :, -1]), axis=0)
    """max_snr_filter: (freq_bins, num_microphones)"""
    Rs_w = np.einsum("kmn,kn->km", Rs, max_snr_filter)
    """Rs_w: (freq_bins, num_microphones)"""
    beta = Rs_w / np.einsum("km,km->k", np.conjugate(max_snr_filter), Rs_w)[:, None]
    """beta: (freq_bins, num_microphones)"""
    w_max_snr = beta[:, None, :] * max_snr_filter[..., None]
    """w_max_snr: (freq_bins, num_microphones, num_microphones)"""
    # フィルタを掛ける
    c_hat = np.einsum("kim,ikt->mkt", np.conjugate(w_max_snr), x)
    """c_hat: (num_microphones, freq_bins, time_frames)"""
    return c_hat

In [7]:
# MWFを実行する
def execute_mwf(x, y, mu):
    """
    x: (num_microphones, freq_bins, time_frames)
    y: (num_microphones, freq_bins, time_frames)
    """
    # 雑音の共分散行列
    Rn = np.average(np.einsum("mkt,nkt->ktmn", y, np.conjugate(y)), axis=1)
    """Rn: (freq_bins, num_microphones, num_microphones)"""
    # 入力共分散行列
    Rs = np.average(np.einsum("mkt,nkt->ktmn", x, np.conjugate(x)), axis=1)
    """Rs: (freq_bins, num_microphones, num_microphones)"""
    # 固有値分解をして半正定行列に変換
    w, v = np.linalg.eigh(Rs)
    Rs_org = Rs.copy()
    w[np.real(w) < 0] = 0 # 固有値が0より小さい場合は0に置き換える
    Rs = np.einsum("kmi,ki,kni->kmn", v, w, np.conjugate(v))
    """Rs: (freq_bins, num_microphones, num_microphones)"""
    # 入力共分散行列
    Rs_muRn = Rs + Rn * mu
    invRs_muRn = np.linalg.pinv(Rs_muRn)
    # フィルタ生成
    W_mwf = np.einsum("kmi,kin->kmn", invRs_muRn, Rs)
    """W_mwf: (freq_bins, num_microphones, num_microphones)"""
    # フィルタを掛ける
    c_hat = np.einsum("kim,ikt->mkt", np.conjugate(W_mwf), x)
    """c_hat: (num_microphones, freq_bins, time_frames)"""
    return c_hat

In [8]:
# SNRを測る
def calculate_snr(target, out):
    """
    target: 目的音 (num_samples, )
    out: 雑音除去後の信号 (num_samples, )
    """
    wave_length = np.minimum(np.shape(target)[0], np.shape(out)[0])
    # 消し残った雑音
    target = target[:wave_length]
    out = out[:wave_length]
    noise = target - out
    snr = 10. * np.log10(np.sum(np.square(target)) / np.sum(np.square(noise)))
    return snr

In [17]:
if __name__ == "__main__":
    # 乱数の種を初期化
    np.random.seed(0)
    # 畳み込みに用いる波形
    clean_wave_files = ["./CMU_ARCTIC/cmu_us_aew/wav/arctic_a0001.wav", "./CMU_ARCTIC/cmu_us_axb/wav/arctic_a0002.wav"]
    # 雑音だけの区間のフレーム数
    n_noise_only = 40000
    # 音源数
    n_sources = len(clean_wave_files)
    # 音声波形の長さを調べる
    n_samples = 0
    # ファイルを読み込む
    for clean_wave_file in clean_wave_files:
        wav = wave.open(clean_wave_file)
        if n_samples<wav.getnframes():
            n_samples=wav.getnframes()
        wav.close()
    clean_data = np.zeros([n_sources, n_samples])

    # ファイルを読み込む
    s = 0
    for clean_wave_file in clean_wave_files:
        wav = wave.open(clean_wave_file)
        data = wav.readframes(wav.getnframes())
        data = np.frombuffer(data, dtype=np.int16)
        data = data/np.iinfo(np.int16).max
        clean_data[s, :wav.getnframes()] = data
        wav.close()
        s = s+1

    # シミュレーションのパラメータ
    n_sim_sources = 2
    # サンプリングレート [Hz]
    sample_rate = 16000
    # フレームサイズ
    N = 1024
    # 周波数の数
    Nk = N / 2 + 1
    # 各ビンの周波数
    freqs = np.arange(0, Nk, 1) * sample_rate / N
    # 音声と雑音の比率 [dB]
    SNR = 10.
    # 部屋の大きさ
    room_dim = np.r_[10.0, 10.0, 10.0]
    # マイクロホンアレイを置く部屋の場所
    mic_array_loc = room_dim / 2 + np.random.randn(3) * 0.1
    # マイクロホンアレイのマイクロホン配置
    mic_alignments = np.array(
            [[x, 0.0, 0.0] for x in np.arange(-0.01, 0.02, 0.02)]
    )
    # マイクロホン数
    n_channels = np.shape(mic_alignments)[0]
    # get the microphone array
    R  = mic_alignments.T + mic_array_loc[:, None]
    """R: (3D-coordinate(x,y,z)=3, num_microphones)"""
    # 部屋を生成する
    room = pa.ShoeBox(room_dim, fs=sample_rate, max_order=17, absorption=0.4)
    room_no_noise = pa.ShoeBox(room_dim, fs=sample_rate, max_order=17, absorption=0.4)
    # 用いるマイクロホンアレイの情報を設置する
    room.add_microphone_array(pa.MicrophoneArray(R, fs=room.fs))
    room_no_noise.add_microphone_array(pa.MicrophoneArray(R, fs=room.fs))
    # 音源の場所
    doas  = np.array(
        [[np.pi/2, 0],
        [np.pi/2, np.pi]]
        )
    # 音源とマイクロホンの距離
    distance = 1.
    source_locations = np.zeros((3, doas.shape[0]), dtype=doas.dtype)
    """source_locations: (xyz, num_sources)"""
    source_locations[0,  :] = np.cos(doas[:, 1]) * np.sin(doas[:, 0]) 
    source_locations[1,  :] = np.sin(doas[:, 1]) * np.sin(doas[:, 0])
    source_locations[2,  :] = np.cos(doas[:, 0])
    source_locations *= distance
    source_locations += mic_array_loc[:, None] # マイクロホンアレイからの相対位置→絶対位置
    # print("clean_data:", np.shape(clean_data))

    # 各音源をシミュレーションに追加する
    for s in range(n_sim_sources):
        clean_data[s] /= np.std(clean_data[s])
        room.add_source(source_locations[:, s], signal=clean_data[s])
        if s == 0:
            room_no_noise.add_source(source_locations[:, s], signal=clean_data[s])

    # シミュレーションを回す
    room.simulate(snr=SNR)
    room_no_noise.simulate(snr=90)

    # 畳み込んだ波形を取得する
    multi_conv_data = room.mic_array.signals
    """multi_conv_data: (num_channels, num_samples)"""
    multi_conv_data_no_noise = room_no_noise.mic_array.signals
    """multi_conv_data_no_noise: (num_channels, num_samples)"""

    # Near仮定に基づくステアリングベクトルを計算: steering_vectors(Nk × Ns × M)
    near_steering_vectors = calculate_steering_vector(R, source_locations, freqs,  is_use_far=False)
    """near_steering_vectors: (freq_bins, num_sources, num_microphones)"""

    # 短時間フーリエ変換
    f, t, stft_data = sp.stft(multi_conv_data, fs=sample_rate, window="hann", nperseg=N)
    """f: (freq_bins,), t: (1,), stft_data:(num_microphones, freq_bins, time_frames)"""

    # 雑音だけの区間のフレーム数
    n_noise_only_frame = np.sum(t < (n_noise_only / sample_rate))

    # 雑音だけのデータ
    noise_data = stft_data[..., :n_noise_only_frame]

    # MWFの雑音の倍率
    mu = 1.0

    # それぞれのフィルタを実行する
    dsbf_out = execute_dsbf(stft_data, near_steering_vectors[:, 0, :])
    mvdr_out = execute_mvdr(stft_data, stft_data, near_steering_vectors[:, 0, :])
    mlbf_out = execute_mvdr(stft_data, noise_data, near_steering_vectors[:, 0, :])
    max_snr_out = execute_max_snr(stft_data, noise_data)
    mwf_out = execute_mwf(stft_data, noise_data, mu)

    # 評価するマイクロホン
    eval_mic_index = 0

    # 時間領域の波形に戻す
    t, dsbf_out = sp.istft(dsbf_out[eval_mic_index], fs=sample_rate, window="hann", nperseg=N)
    t, mvdr_out = sp.istft(mvdr_out[eval_mic_index], fs=sample_rate, window="hann", nperseg=N)
    t, mlbf_out = sp.istft(mlbf_out[eval_mic_index], fs=sample_rate, window="hann", nperseg=N)
    t, max_snr_out = sp.istft(max_snr_out[eval_mic_index], fs=sample_rate, window="hann", nperseg=N)
    t, mwf_out = sp.istft(mwf_out[eval_mic_index], fs=sample_rate, window="hann", nperseg=N)

    # SNRを測る
    snr_pre = calculate_snr(multi_conv_data_no_noise[eval_mic_index, n_noise_only:], multi_conv_data[eval_mic_index, n_noise_only:])
    snr_dsbf_post = calculate_snr(multi_conv_data_no_noise[eval_mic_index, n_noise_only:], dsbf_out[n_noise_only:])
    snr_mvdr_post = calculate_snr(multi_conv_data_no_noise[eval_mic_index, n_noise_only:], mvdr_out[n_noise_only:])
    snr_mlbf_post = calculate_snr(multi_conv_data_no_noise[eval_mic_index, n_noise_only:], mlbf_out[n_noise_only:])
    snr_max_snr_post = calculate_snr(multi_conv_data_no_noise[eval_mic_index, n_noise_only:], max_snr_out[n_noise_only:])
    snr_mwf_post = calculate_snr(multi_conv_data_no_noise[eval_mic_index, n_noise_only:], mwf_out[n_noise_only:])
    print("result ΔSNR [dB]")
    print("DSBF:{:.2f}".format(snr_dsbf_post-snr_pre))
    print("MVDR:{:.2f}".format(snr_mvdr_post-snr_pre))
    print("MLBF:{:.2f}".format(snr_mlbf_post-snr_pre))
    print("MaxSNR:{:.2f}".format(snr_max_snr_post-snr_pre))
    print("MWF:{:.2f}".format(snr_mwf_post-snr_pre))

beta: (513, 2)
w_max_snr: (513, 2, 2)
c_hat: (2, 513, 139)
result ΔSNR [dB]
DSBF:1.59
MVDR:1.36
MLBF:0.75
MaxSNR:-0.36
MWF:2.18
